# 10 · Parâmetros de ordem S² das ligações N–H (validação por RMN)

| | |
|---|---|
| **Autor** | Patrick Allan dos Santos Faustino |
| **Projeto** | Revisão tutorial de dinâmica molecular — xilanase GH11 (PDB 1XNB) |
| **Versão** | v2026.11 — notebook autocontido |

**Pergunta.** A dinâmica rápida do esqueleto na simulação tem a mesma amplitude que a relaxação de ¹⁵N mede em solução?

O **parâmetro de ordem generalizado** $S^2$ (Lipari & Szabo, *J. Am. Chem. Soc.* **1982**, 104,
4546) mede a restrição angular do vetor N–H de cada ligação peptídica: 1 = rígido, 0 = livre. Na
DM, é o platô da autocorrelação de $P_2$ depois de removida a rotação global:

$$S^2 = \frac{3}{2}\sum_{i,j}\langle \mu_i \mu_j \rangle^2 - \frac{1}{2},$$

com $\boldsymbol\mu$ o vetor unitário N→H e $i, j \in \{x, y, z\}$.

**Dado experimental:** Connelly, Withers & McIntosh (*Protein Sci.* **2000**, 9, 512) mediram $T_1$,
$T_2$ e NOE de ¹⁵N em ~120 dos 178 grupos peptídicos da BCX livre. Pelo *model-free*, o $S^2$ médio
é **0,86 ± 0,04** e o tempo de correlação global $\tau_m$ é **9,2 ± 0,2 ns**.

* **Janela.** O $S^2$ de RMN só "vê" movimentos mais rápidos que $\tau_m$: a média sobre a produção
  inteira mistura movimentos lentos e subestima $S^2$. Aqui o $S^2$ é calculado em blocos de
  duração ≈ $\tau_m$ (`BLOCO_NS`) e depois tirada a média (prática usual na comparação DM × RMN).
* **Dados por resíduo.** Para comparar resíduo a resíduo, digite os valores do artigo num CSV
  (`residuo,S2[,erro]`) e aponte `ARQUIVO_S2_EXP`. Sem ele, compara-se o $S^2$ médio.

> **Pré-processamento** (a PBC é tratada no GROMACS, não em Python):
> ```bash
> gmx trjconv -s md_300ns.tpr -f md_300ns.xtc -o md_whole.xtc -pbc whole
> gmx trjconv -s md_300ns.tpr -f md_whole.xtc -o md_300ns_mol.xtc -pbc mol -center -ur compact
> ```

**Blocos:** 1 bibliotecas · 2 configuração e dados · 3 cálculos · 4 estatística ·
5 gráfico (matplotlib) · 6 gráfico interativo (Plotly) · 7 exportação.

## 1 · Bibliotecas

In [ ]:
import sys
import platform
from datetime import datetime
from pathlib import Path
import re

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mdtraj as md
import MDAnalysis as mda
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

INICIO = datetime.now()
VERSOES = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "SciPy": scipy.__version__,
    "Matplotlib": matplotlib.__version__,
    "Plotly": plotly.__version__,
    "MDTraj": md.__version__,
    "MDAnalysis": mda.__version__,
}
print(f"Data e hora da execução: {INICIO:%d/%m/%Y %H:%M:%S}")
print(f"Sistema operacional: {platform.platform()}")
df_versoes = pd.DataFrame(list(VERSOES.items()), columns=["biblioteca", "versao"])
df_versoes

## 2 · Configuração da análise e carregamento dos dados

In [ ]:
# ── Onde estão as simulações ────────────────────────────────────────────────
PASTA_BASE = Path("..")             # pasta que contém rep1/, rep2/, ...   <- AJUSTE
SISTEMA = "xilanase_1XNB"           # nome da pasta de resultados
REPLICAS = [1, 2, 3, 4, 5]          # uma réplica só também funciona: [1]
PREFIXO_MD = "md_300ns"             # -deffnm da produção


def pasta_replica(n):
    """Pasta da réplica n."""
    return PASTA_BASE / f"rep{n}"

ARQUIVO_TOP = f"{PREFIXO_MD}.gro"            # sistema inteiro
ARQUIVO_TRAJ = f"{PREFIXO_MD}_mol.xtc"        # sistema inteiro, -pbc mol -center
ARQUIVO_TPR = f"{PREFIXO_MD}.tpr"            # elementos pelo nº atômico (MDAnalysis >= 2.11); None = pelo nome
ARQUIVOS = {"topologia": ARQUIVO_TOP, "trajetória": ARQUIVO_TRAJ, **({"tpr": ARQUIVO_TPR} if ARQUIVO_TPR else {})}

# ── Amostragem ──────────────────────────────────────────────────────────────
DT_ANALISE_PS = 20.0            # S² precisa de muitos frames por bloco (500 frames em 10 ns)
DESCARTE_NS = 0.0
NIVEL_CONFIANCA = 0.95

# ── Análise ─────────────────────────────────────────────────────────────────
NOMES_H = ("H", "HN")           # nome do H amida (AMBER/GROMACS: H; CHARMM: HN)
BLOCO_NS = 10.0                 # ≈ τm = 9,2 ns da BCX (Connelly et al. 2000)
ITERACOES_MEDIA = 3             # alinhamento (Cα) à estrutura média de cada réplica
S2_MEDIO_EXP, U_S2_MEDIO_EXP = 0.86, 0.04      # BCX livre, Connelly et al., Protein Sci. 2000, 9, 512
ARQUIVO_S2_EXP = None           # CSV com colunas residuo, S2 (e, opcional, erro)
DESLOCAMENTO_RESIDUO = 0        # somado ao nº do resíduo do CSV experimental

# ── Saída ───────────────────────────────────────────────────────────────────
DIR_SAIDA = Path("resultados") / SISTEMA / "10_parametros_ordem_s2"
PREFIXO = "s2"
TITULO = "10 · Parâmetros de ordem S²"

In [ ]:
linhas = []
for n in REPLICAS:
    for papel, nome in ARQUIVOS.items():
        caminho = pasta_replica(n) / nome
        linhas.append({"replica": f"rep{n}", "arquivo": papel, "caminho": str(caminho),
                       "existe": caminho.exists(),
                       "tamanho_MB": round(caminho.stat().st_size / 1024 ** 2, 1) if caminho.exists() else np.nan})
df_arquivos = pd.DataFrame(linhas)
display(df_arquivos)
if not df_arquivos["existe"].all():
    raise FileNotFoundError("Arquivos ausentes (veja a tabela acima). Ajuste RAIZ/SISTEMA/REPLICAS.")

In [ ]:
MAPA_RESNAMES = {"HIE": "HIS", "HID": "HIS", "HIP": "HIS", "HSD": "HIS", "HSE": "HIS",
                 "HSP": "HIS", "CYX": "CYS", "CYM": "CYS", "LYN": "LYS", "ASH": "ASP",
                 "GLH": "GLU"}


def tpr_de(n):
    """Caminho do .tpr da réplica n (None se ARQUIVO_TPR = None)."""
    return pasta_replica(n) / ARQUIVO_TPR if ARQUIVO_TPR else None


def elementos_do_tpr(tpr, top, arquivo):
    """Elemento de cada átomo da topologia, pelo número atômico gravado no .tpr.

    O .gro pode ser o sistema inteiro ou um subconjunto na MESMA ordem (o _prot.gro).
    O casamento é resíduo a resíduo, pela sequência de nomes de átomos: o MDAnalysis
    renumera os resíduos do .tpr a partir de 1, então o nº do resíduo (Leu4...) não serve.
    """
    u = mda.Universe(str(tpr))
    if not hasattr(u.atoms, "elements"):
        raise ValueError("o .tpr não trouxe números atômicos")
    if str(arquivo).endswith(".gro"):            # nomes crus (o MDTraj renomeia OW -> O etc.)
        linhas = Path(arquivo).read_text().splitlines()[2:2 + top.n_atoms]
        nomes = [l[10:15].strip() for l in linhas]
    else:
        nomes = [a.name for a in top.atoms]
    cortes = np.flatnonzero(np.diff(u.atoms.resindices)) + 1
    residuos_tpr = list(zip(np.split(u.atoms.names, cortes), np.split(u.atoms.elements, cortes)))
    elementos, j = [], 0
    for residuo in top.residues:
        alvo = [nomes[a.index] for a in residuo.atoms]
        while j < len(residuos_tpr) and list(residuos_tpr[j][0]) != alvo:
            j += 1
        if j == len(residuos_tpr):
            raise ValueError(f"{residuo} não casa com o .tpr (o .gro não é um subconjunto ordenado dele)")
        elementos += list(residuos_tpr[j][1])
        j += 1
    return elementos


def ler_topologia(arquivo, tpr=None):
    """Topologia MDTraj de um .gro/.pdb, com os elementos certos.

    * Com `tpr` (MDAnalysis >= 2.11 lê o .tpr do GROMACS 2026): o elemento vem do número
      atômico gravado no .tpr — a mesma informação que o GROMACS usa.
    * Sem `tpr` (ou se a leitura falhar): o MDTraj deduz o elemento pelo nome do átomo, e o
      FE do heme viraria flúor (19 u em vez de 55,85 u); ele é corrigido e um aviso é impresso.
    * Resíduos com nome de protonação (HIE, CYM...) recebem o nome padrão, para que
      `protein` e `resname` funcionem.
    """
    top = md.load_topology(str(arquivo))
    for residuo in top.residues:
        residuo.name = MAPA_RESNAMES.get(residuo.name.upper(), residuo.name)
    if tpr is not None and Path(tpr).exists():
        try:
            elementos = elementos_do_tpr(tpr, top, arquivo)
            for atomo, simbolo in zip(top.atoms, elementos):
                atomo.element = md.element.get_by_symbol(simbolo.capitalize()) if simbolo else md.element.virtual
            return top
        except Exception as erro:
            print(f"[aviso] .tpr não usado ({erro}); elementos deduzidos pelo nome do átomo.")
    for atomo in top.atoms:
        if atomo.residue.name == "HEM" and atomo.name.upper() == "FE":
            atomo.element = md.element.iron
    return top


def selecionar(top, consulta, n=None):
    """top.select com erro explícito se vazio (ou se não tiver n átomos)."""
    idx = top.select(consulta)
    if len(idx) == 0:
        raise ValueError(f"Seleção vazia: '{consulta}' (no MDTraj o nº do resíduo é `resSeq`).")
    if n is not None and len(idx) != n:
        raise ValueError(f"'{consulta}' casou com {len(idx)} átomos (esperado {n}).")
    return idx


def intervalo_do_arquivo_ps(xtc, top):
    """Δt entre frames gravados no .xtc (lê só os 2 primeiros frames)."""
    for lote in md.iterload(str(xtc), top=top, chunk=2, atom_indices=[0]):
        return float(np.diff(lote.time)[0])


def estride(xtc, top, dt_alvo_ps):
    """Passo de leitura que leva o Δt do arquivo ao Δt de análise."""
    dt = intervalo_do_arquivo_ps(xtc, top)
    passo = max(1, int(round(dt_alvo_ps / dt)))
    if dt_alvo_ps < dt:
        print(f"[aviso] Δt pedido {dt_alvo_ps:g} ps é menor que o do arquivo ({dt:g} ps).")
    elif abs(passo * dt - dt_alvo_ps) > 1e-6 * dt_alvo_ps:
        print(f"[aviso] Δt {dt_alvo_ps:g} ps não é múltiplo de {dt:g} ps; usando {passo * dt:g} ps.")
    return passo, dt


def producao(t_ns):
    """Máscara dos frames de produção (depois de DESCARTE_NS)."""
    t_ns = np.asarray(t_ns, dtype=float)
    mascara = t_ns >= t_ns[0] + DESCARTE_NS
    if not mascara.any():
        raise ValueError("DESCARTE_NS é maior que a trajetória.")
    return mascara


def carregar(n, atomos=None, dt_ps=None):
    """Lê a réplica n (só os átomos `atomos`, índices da topologia) -> (traj, t_ns)."""
    xtc, top = pasta_replica(n) / ARQUIVO_TRAJ, ler_topologia(pasta_replica(n) / ARQUIVO_TOP, tpr_de(n))
    passo, dt = estride(xtc, top, dt_ps or DT_ANALISE_PS)
    traj = md.load(str(xtc), top=top, stride=passo,
                   atom_indices=None if atomos is None else np.unique(atomos))
    t_ns = np.asarray(traj.time, dtype=float) / 1000.0
    if not np.all(np.diff(t_ns) > 0):
        raise ValueError(f"rep{n}: tempos não crescentes (use `gmx trjcat -cat`).")
    return traj, t_ns



top = ler_topologia(pasta_replica(REPLICAS[0]) / ARQUIVO_TOP, tpr_de(REPLICAS[0]))
pares, residuos, nomes = [], [], []
for r in top.residues:
    if not r.is_protein:
        continue
    n = [a.index for a in r.atoms if a.name == "N"]
    h = [a.index for a in r.atoms if a.name in NOMES_H]
    if n and h:                                  # Pro e N-terminal (H1/H2/H3) ficam de fora
        pares.append((n[0], h[0])); residuos.append(r.resSeq); nomes.append(r.name.capitalize())
idx_ca = selecionar(top, "protein and name CA")
if not pares:
    raise ValueError(f"Nenhum par N–H com H em {NOMES_H}: a trajetória tem os hidrogênios?")
idx = np.unique(np.r_[np.ravel(pares), idx_ca])
loc_n, loc_h = np.searchsorted(idx, [p[0] for p in pares]), np.searchsorted(idx, [p[1] for p in pares])
loc_ca = np.searchsorted(idx, idx_ca)
residuos, nomes = np.array(residuos), np.array(nomes)
print(f"{len(pares)} vetores N–H (de {sum(r.is_protein for r in top.residues)} resíduos)")
trajs = {f"rep{n}": carregar(n, idx) for n in REPLICAS}

df_exp = pd.DataFrame()
if ARQUIVO_S2_EXP and Path(ARQUIVO_S2_EXP).exists():
    df_exp = pd.read_csv(ARQUIVO_S2_EXP).rename(columns={"S2": "s2_exp", "erro": "erro_exp"})
    df_exp["residuo"] = df_exp["residuo"] + DESLOCAMENTO_RESIDUO
    print(f"S² experimental: {len(df_exp)} resíduos de {ARQUIVO_S2_EXP}")
else:
    print(f"S² experimental por resíduo ausente — compara-se o médio ({S2_MEDIO_EXP} ± {U_S2_MEDIO_EXP}).")
df_trajetorias = pd.DataFrame([{
    "replica": rep, "frames": tr.n_frames, "atomos": tr.n_atoms,
    "dt_ps": round(1000 * float(np.median(np.diff(t))), 3),
    "inicio_ns": t[0], "fim_ns": t[-1],
    "frames_producao": int(producao(t).sum()),
    "memoria_MB": round(tr.xyz.nbytes / 1024 ** 2, 1)}
    for rep, (tr, t) in trajs.items()])
df_trajetorias

## 3 · Cálculos

Cada réplica é alinhada (Cα) à **própria estrutura média** — remove translação e rotação global,
como a análise *model-free* separa a difusão rotacional. Depois, em cada bloco de `BLOCO_NS`:
o tensor $\langle \boldsymbol\mu \boldsymbol\mu^T\rangle$ de cada N–H e
$S^2 = \tfrac{3}{2}\lVert\langle \boldsymbol\mu \boldsymbol\mu^T\rangle\rVert_F^2 - \tfrac{1}{2}$.

In [ ]:
def alinhar_a_media(traj, idx, iteracoes=3):
    trabalho = traj[:]
    ref = trabalho[0]
    for _ in range(iteracoes):
        trabalho.superpose(ref, atom_indices=idx, ref_atom_indices=idx)
        ref = trabalho[0]
        ref.xyz = trabalho.xyz.mean(axis=0, keepdims=True)
    trabalho.superpose(ref, atom_indices=idx, ref_atom_indices=idx)
    return trabalho


def s2_de(mu):
    """mu: (F, N, 3) unitários -> S² (N,)."""
    tensor = np.nanmean(mu[:, :, :, None] * mu[:, :, None, :], axis=0)      # ignora frames inválidos (NaN)
    return 1.5 * (tensor ** 2).sum(axis=(1, 2)) - 0.5


quadros, por_bloco = [], []
for rep, (traj, t_ns) in trajs.items():
    alinhada = alinhar_a_media(traj[producao(t_ns)], loc_ca, ITERACOES_MEDIA)
    v = alinhada.xyz[:, loc_h, :].astype(np.float64) - alinhada.xyz[:, loc_n, :]
    norma = np.linalg.norm(v, axis=2, keepdims=True)
    ruins = norma[..., 0] < 0.05                       # N–H tem ~0,101 nm: menos que isso é geometria quebrada
    if ruins.any():
        print(f"[aviso] {rep}: {int(ruins.sum())} vetor(es) N–H com < 0,05 nm (resíduos "
              f"{sorted(int(r) for r in set(residuos[np.flatnonzero(ruins.any(axis=0))]))[:5]}) — ignorados.")
    mu = np.where(ruins[..., None], np.nan, v / np.where(norma > 0, norma, 1.0))
    t = t_ns[producao(t_ns)]
    n_blocos = max(1, int((t[-1] - t[0]) // BLOCO_NS))
    blocos = [s2_de(mu[f]) for f in np.array_split(np.arange(mu.shape[0]), n_blocos)]
    for b, s2 in enumerate(blocos, start=1):
        por_bloco.append(pd.DataFrame({"replica": rep, "bloco": b, "residuo": residuos, "s2": s2}))
    quadros.append(pd.DataFrame({"replica": rep, "residuo": residuos, "nome": nomes,
                                 "s2": np.mean(blocos, axis=0), "s2_producao_inteira": s2_de(mu),
                                 "n_blocos": n_blocos}))
    print(f"[{rep}] {n_blocos} bloco(s) de ~{(t[-1] - t[0]) / n_blocos:.1f} ns")
df_s2 = pd.concat(quadros, ignore_index=True)
df_blocos = pd.concat(por_bloco, ignore_index=True)
df_s2.groupby("replica", sort=False)[["s2", "s2_producao_inteira"]].mean()

## 4 · Estatística

Por resíduo: média ± SEM entre réplicas (com 1 réplica, entre blocos). $S^2$ médio da proteína:
um valor por réplica → média ± SEM entre réplicas, comparado com 0,86 ± 0,04 em unidades da
incerteza combinada. Com o CSV por resíduo: Pearson, Spearman e RMSD entre os perfis (só os
resíduos medidos).

In [ ]:
def ineficiencia_estatistica(x, t_minimo=3):
    """g = 1 + 2·τ_int (em frames) — Chodera et al., JCTC 2007, 3, 26.

    ACF não enviesada (FFT), somada até o 1º C(t) <= 0 depois de t_minimo, com o
    peso (1 − t/N) aplicado UMA vez. Igual a
    pymbar.timeseries.statistical_inefficiency(fast=False).
    """
    x = np.asarray(x, dtype=float)
    n = x.size
    if n < 3 or np.allclose(x, x[0]):
        return 1.0
    d = x - x.mean()
    var = float(d @ d) / n
    tamanho = 1 << (2 * n - 1).bit_length()
    f = np.fft.rfft(d, tamanho)
    acf = np.fft.irfft(f * np.conjugate(f), tamanho)[:n] / ((n - np.arange(n)) * var)
    g = 1.0
    for t in range(1, n - 1):
        if acf[t] <= 0.0 and t > t_minimo:
            break
        g += 2.0 * acf[t] * (1.0 - t / n)
    return float(max(g, 1.0))


def t_student(graus_liberdade):
    return stats.t.ppf(0.5 + NIVEL_CONFIANCA / 2, df=max(graus_liberdade, 1))


def resumo_replica(y, dt_ns, observavel, replica, unidade=""):
    """Média, DP, g, N_ef e SEM corrigido de UMA série (uma réplica)."""
    y = np.asarray(y, dtype=float)
    g = ineficiencia_estatistica(y)
    n_ef = max(y.size / g, 2.0)
    dp = float(np.std(y, ddof=1))
    sem = dp / np.sqrt(n_ef)
    media = float(np.mean(y))
    return {"observavel": observavel, "replica": replica, "unidade": unidade,
            "n": y.size, "media": media, "desvio_padrao": dp, "sem": sem,
            "ic_inferior": media - t_student(n_ef - 1) * sem,
            "ic_superior": media + t_student(n_ef - 1) * sem,
            "minimo": float(y.min()), "maximo": float(y.max()),
            "g": g, "tau_int_ns": (g - 1.0) / 2.0 * dt_ns, "n_efetivo": n_ef}


def entre_replicas(medias, observavel, unidade=""):
    """A réplica é a unidade estatística: média ± SEM das médias das réplicas."""
    medias = np.asarray(medias, dtype=float)
    k = medias.size
    media = float(medias.mean())
    dp = float(np.std(medias, ddof=1)) if k > 1 else np.nan
    sem = dp / np.sqrt(k) if k > 1 else np.nan
    return {"observavel": observavel, "replica": "entre_replicas", "unidade": unidade,
            "n": k, "media": media, "desvio_padrao": dp, "sem": sem,
            "ic_inferior": media - t_student(k - 1) * sem if k > 1 else np.nan,
            "ic_superior": media + t_student(k - 1) * sem if k > 1 else np.nan,
            "minimo": float(medias.min()), "maximo": float(medias.max()),
            "g": np.nan, "tau_int_ns": np.nan, "n_efetivo": float(k)}


def estatisticas_series(df_series, unidades):
    """Por réplica (só produção) + linha 'entre_replicas' quando há 2 ou mais."""
    linhas = []
    prod = df_series[df_series["fase"] == "producao"]
    for obs, bloco in prod.groupby("observavel", sort=False):
        por_rep = [resumo_replica(g["valor"], float(np.median(np.diff(g["tempo_ns"]))),
                                  obs, rep, unidades.get(obs, ""))
                   for rep, g in bloco.groupby("replica", sort=False)]
        linhas += por_rep
        if len(por_rep) > 1:
            linhas.append(entre_replicas([l["media"] for l in por_rep], obs, unidades.get(obs, "")))
    return pd.DataFrame(linhas)


def referencia(df_stats, observavel):
    """Linha 'entre_replicas' do observável (ou a da única réplica)."""
    bloco = df_stats[df_stats["observavel"] == observavel]
    entre = bloco[bloco["replica"] == "entre_replicas"]
    return (entre if len(entre) else bloco).iloc[0]


def formatar(media, erro):
    """'média ± erro' com 2 algarismos significativos no erro (GUM, JCGM 100:2008 §7.2.6)."""
    if not (np.isfinite(erro) and erro > 0):
        return f"{media:.4g}"
    casas = max(0, 1 - int(np.floor(np.log10(abs(erro)))))
    return f"{media:.{casas}f} ± {erro:.{casas}f}"


_SOBRESCRITO = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")


def unidade_mpl(unidade):
    """Unidade para o matplotlib: '⁻³' vira mathtext ($^{-3}$). O sinal sobrescrito '⁻' (U+207B)
    falta em várias fontes (a Aptos Serif, inclusive) e sai como um quadrado na figura."""
    return re.sub(r"⁻([⁰¹²³⁴⁵⁶⁷⁸⁹]+)",
                  lambda m: "$^{-" + m.group(1).translate(_SOBRESCRITO) + "}$", unidade or "")


def rotulo_media(ref):
    """Rótulo dos painéis: 'média ± EP (N réplicas): valor ± erro unidade'.

    EP = erro padrão da média: entre as médias das réplicas (a réplica é a unidade estatística);
    com uma réplica só, o SEM da série corrigido pela ineficiência estatística."""
    n = ref.get("n", np.nan)
    origem = (f"{int(n)} réplicas" if ref.get("replica") == "entre_replicas" and pd.notna(n)
              else "1 réplica")
    return (f"média ± EP ({origem}): {formatar(ref['media'], ref['sem'])} "
            f"{unidade_mpl(ref.get('unidade', ''))}").strip()

In [ ]:
tab = df_s2.pivot(index="residuo", columns="replica", values="s2")
k = tab.shape[1]
if k > 1:
    media, sem, gl, fonte_erro = tab.mean(axis=1), tab.std(axis=1, ddof=1) / np.sqrt(k), k - 1, f"entre {k} réplicas"
else:
    bl = df_blocos.pivot(index="residuo", columns="bloco", values="s2")
    media, sem, gl = tab.iloc[:, 0], bl.std(axis=1, ddof=1) / np.sqrt(bl.shape[1]), bl.shape[1] - 1
    fonte_erro = f"entre {bl.shape[1]} blocos (1 réplica)"
df_perfil = pd.DataFrame({"residuo": tab.index, "nome": pd.Series(nomes, index=residuos).reindex(tab.index).to_numpy(),
                          "s2_medio": media.to_numpy(), "sem": sem.to_numpy(),
                          "ic_inferior": (media - t_student(gl) * sem).to_numpy(),
                          "ic_superior": (media + t_student(gl) * sem).to_numpy()})
if len(df_exp):
    df_perfil = df_perfil.merge(df_exp, on="residuo", how="left")

medidos = df_perfil["s2_exp"].notna() if "s2_exp" in df_perfil else np.ones(len(df_perfil), bool)
medias_rep = tab[np.asarray(medidos)].mean(axis=0)
linhas = [{"observavel": "S² médio", "replica": r, "unidade": "", "media": v, "sem": np.nan} for r, v in medias_rep.items()]
if k > 1:
    linhas.append(entre_replicas(medias_rep, "S² médio", ""))
df_stats = pd.DataFrame(linhas)
ref = referencia(df_stats, "S² médio")
u_comb = np.hypot(np.nan_to_num(ref["sem"]), U_S2_MEDIO_EXP)
df_comparacao = pd.DataFrame([{"grandeza": "S² médio", "simulado": ref["media"], "incerteza_sim": ref["sem"],
                               "experimental": S2_MEDIO_EXP, "incerteza_exp": U_S2_MEDIO_EXP,
                               "desvio": ref["media"] - S2_MEDIO_EXP, "desvio_em_incertezas": (ref["media"] - S2_MEDIO_EXP) / u_comb}])
if "s2_exp" in df_perfil:
    q = df_perfil.dropna(subset=["s2_exp"])
    df_comparacao = pd.concat([df_comparacao, pd.DataFrame([{
        "grandeza": "perfil por resíduo", "n_residuos": len(q), "pearson_r": stats.pearsonr(q["s2_medio"], q["s2_exp"])[0],
        "spearman_rho": stats.spearmanr(q["s2_medio"], q["s2_exp"])[0],
        "rmsd": float(np.sqrt(((q["s2_medio"] - q["s2_exp"]) ** 2).mean()))}])], ignore_index=True)
if len(REPLICAS) == 1:
    print("[1 réplica] Não há linha 'entre_replicas': o erro é o SEM da própria série, "
          "corrigido pela ineficiência estatística (subestima a incerteza de observáveis lentos).")
print(f"S² médio (DM) = {formatar(ref['media'], ref['sem'])} | experimental {S2_MEDIO_EXP} ± {U_S2_MEDIO_EXP} "
      f"| desvio = {df_comparacao['desvio_em_incertezas'][0]:+.1f} incertezas")
print(f"Sem blocos (produção inteira): {df_s2['s2_producao_inteira'].mean():.3f} — a diferença mostra o peso dos movimentos lentos.")
df_comparacao

## 5 · Gráfico (matplotlib)

In [ ]:
# ── Estilo geral (vale para todas as figuras deste notebook) ────────────────
FONTE = "Arial"                 # se não estiver instalada, usa DejaVu Serif
TAMANHO_FONTE = 10                     # pt
CORES = ["#003f5c", "#bc5090", "#ffa600", "#58508d", "#ff6361"]   # uma por réplica
COR_MEDIA = "#333333"                 # tracejado da média
COR_REFERENCIA = "#d62728"            # linhas de referência e faixa do descarte

if FONTE not in {f.name for f in font_manager.fontManager.ttflist}:
    FONTE = "DejaVu Serif"
plt.rcParams.update({
    "font.family": FONTE, "font.size": TAMANHO_FONTE,
    "axes.labelsize": TAMANHO_FONTE, "axes.titlesize": TAMANHO_FONTE,
    "xtick.labelsize": TAMANHO_FONTE - 1, "ytick.labelsize": TAMANHO_FONTE - 1,
    "legend.fontsize": TAMANHO_FONTE - 1, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.dpi": 120,
    "pdf.fonttype": 42,                # fontes embutidas como TrueType (editáveis)
    "svg.fonttype": "none",            # texto do SVG continua texto
})

In [ ]:
# ── Layout desta figura ─────────────────────────────────────────────────────
LARGURA, ALTURA = 6.3, 3.25
ESPESSURA = 0.6
OPACIDADE_REPLICAS = 0.7
MOSTRAR_REPLICAS = True
MOSTRAR_HISTOGRAMA = True      # distribuição do S² médio por resíduo à direita
BINS = 30
LIMITES_Y = (0.0, 1.0)

# ── Gráfico ─────────────────────────────────────────────────────────────────
tem_exp = "s2_exp" in df_perfil
proporcoes = [4] + ([1] if MOSTRAR_HISTOGRAMA else []) + ([1.6] if tem_exp else [])
fig, eixos = plt.subplots(1, len(proporcoes), figsize=(LARGURA, ALTURA), layout="constrained", squeeze=False,
                          gridspec_kw={"width_ratios": proporcoes})
ax = eixos[0, 0]
x = df_perfil["residuo"].to_numpy()
y = df_perfil["s2_medio"].to_numpy()
ep = df_perfil["sem"].to_numpy()
if MOSTRAR_REPLICAS and k > 1:
    for i, (rep, g) in enumerate(df_s2.groupby("replica", sort=False)):
        ax.plot(g["residuo"], g["s2"], color=CORES[i % len(CORES)], lw=ESPESSURA, alpha=OPACIDADE_REPLICAS, label=rep)
ax.fill_between(x, y - ep, y + ep, color=COR_MEDIA, alpha=0.25, lw=0)          # ± EP por resíduo
ax.plot(x, y, color=COR_MEDIA, lw=1.0)
if tem_exp:
    ax.errorbar(x, df_perfil["s2_exp"], yerr=df_perfil.get("erro_exp"), fmt="o", ms=2.5, color=COR_REFERENCIA, lw=0.6)
ax.axhline(ref["media"], color=COR_MEDIA, ls="--", lw=0.8)
ax.axhline(S2_MEDIO_EXP, color=COR_REFERENCIA, ls=":", lw=0.8)
ax.annotate(f"RMN {S2_MEDIO_EXP} ± {U_S2_MEDIO_EXP}", (1, S2_MEDIO_EXP), xycoords=("axes fraction", "data"),
            xytext=(0, -2), textcoords="offset points", ha="right", va="top", color=COR_REFERENCIA,
            fontsize=TAMANHO_FONTE - 2, bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
ax.set(xlabel="Resíduo", ylabel="S$^2$ (N–H)", ylim=LIMITES_Y, xlim=(x.min(), x.max()))
ax.set_title("A", loc="left", fontweight="bold")
ax.set_title(rotulo_media(ref), loc="right", fontsize=TAMANHO_FONTE - 1)
if MOSTRAR_HISTOGRAMA:
    axh = eixos[0, 1]
    axh.hist(y[np.isfinite(y)], bins=BINS, orientation="horizontal", color=CORES[0], alpha=0.4)
    axh.axhline(ref["media"], color=COR_MEDIA, ls="--", lw=0.8)
    axh.axhline(S2_MEDIO_EXP, color=COR_REFERENCIA, ls=":", lw=0.8)
    axh.set(ylim=LIMITES_Y, xticks=[], xlabel="Distribuição")
    axh.tick_params(labelleft=False)
    axh.spines[["top", "right"]].set_visible(False)
if tem_exp:
    q = df_perfil.dropna(subset=["s2_exp"])
    axd = eixos[0, -1]
    axd.scatter(q["s2_exp"], q["s2_medio"], s=8, color=COR_MEDIA, alpha=0.7, lw=0)
    axd.plot([0, 1], [0, 1], color=COR_MEDIA, ls="--", lw=0.7)
    axd.set(xlabel="S$^2$ RMN", ylabel="S$^2$ DM", xlim=(0.4, 1), ylim=(0.4, 1), aspect="equal")
    axd.set_title("B", loc="left", fontweight="bold")
if MOSTRAR_REPLICAS and k > 1:
    fig.legend(*ax.get_legend_handles_labels(), loc="outside upper center", ncol=k, frameon=True)
fig_s2 = fig
plt.show()


## 6 · Gráfico interativo (Plotly)

In [ ]:
# ── Layout ──────────────────────────────────────────────────────────────────
ALTURA_PX = 420                # pixels
LARGURA_PX = None              # None = largura da página
TEMPLATE = "simple_white"
TAMANHO_FONTE_PX = 13

# ── Gráfico ─────────────────────────────────────────────────────────────────
# legenda: só as réplicas; a média (linha escura), o IC e a RMN estão rotulados no gráfico/hover
x = df_perfil["residuo"].to_numpy()
fig_i = go.Figure()
fig_i.add_trace(go.Scatter(x=np.r_[x, x[::-1]], y=np.r_[df_perfil["ic_superior"], df_perfil["ic_inferior"][::-1]],
                           fill="toself", fillcolor="rgba(51,51,51,0.15)", line=dict(width=0), hoverinfo="skip",
                           showlegend=False))
for i, (rep, g) in enumerate(df_s2.groupby("replica", sort=False)):
    fig_i.add_trace(go.Scatter(x=g["residuo"], y=g["s2"], mode="lines", name=rep,
                               line=dict(color=CORES[i % len(CORES)], width=1)))
fig_i.add_trace(go.Scatter(x=x, y=df_perfil["s2_medio"], mode="lines", name="média", showlegend=False,
                           line=dict(color=COR_MEDIA, width=2), customdata=np.c_[df_perfil["nome"], df_perfil["sem"]],
                           hovertemplate="média — %{customdata[0]}%{x}: S² = %{y:.3f} ± %{customdata[1]:.3f}<extra></extra>"))
if "s2_exp" in df_perfil:
    fig_i.add_trace(go.Scatter(x=x, y=df_perfil["s2_exp"], mode="markers", name="RMN", showlegend=False,
                               marker=dict(color=COR_REFERENCIA, size=6),
                               hovertemplate="RMN — resíduo %{x}: S² = %{y:.3f}<extra></extra>"))
fig_i.add_trace(go.Scatter(x=[x.min(), x.max()], y=[ref["media"]] * 2, mode="lines", showlegend=False, hoverinfo="skip",
                           line=dict(color=COR_MEDIA, dash="dash", width=1.5)))
fig_i.add_annotation(x=x.min(), y=ref["media"], text=f"DM média ± EP: {formatar(ref['media'], ref['sem'])}",
                     showarrow=False, xanchor="left", yanchor="bottom", bgcolor="rgba(255,255,255,0.7)")
fig_i.add_trace(go.Scatter(x=[x.min(), x.max()], y=[S2_MEDIO_EXP] * 2, mode="lines", showlegend=False, hoverinfo="skip",
                           line=dict(color=COR_REFERENCIA, dash="dot", width=1.5)))
fig_i.add_annotation(x=x.max(), y=S2_MEDIO_EXP, text=f"RMN média {S2_MEDIO_EXP} ± {U_S2_MEDIO_EXP}", showarrow=False,
                     xanchor="right", yanchor="top", font=dict(color=COR_REFERENCIA))
fig_i.add_annotation(x=0, y=0, xref="x domain", yref="y domain", xanchor="left", yanchor="bottom", showarrow=False,
                     text=f"linha escura: média ± IC {NIVEL_CONFIANCA:.0%} ({fonte_erro})"
                          + ("; pontos vermelhos: RMN" if "s2_exp" in df_perfil else ""))
fig_i.update_xaxes(title_text="Resíduo")
fig_i.update_yaxes(title_text="S² (N–H)", range=[0, 1])
fig_i.update_layout(template=TEMPLATE, height=ALTURA_PX, width=LARGURA_PX, legend_title_text="Réplica",
                    font=dict(family=FONTE, size=TAMANHO_FONTE_PX))
fig_interativo = fig_i
fig_i.show()

## 7 · Exportação

Tudo vai para `DIR_SAIDA`, separado por formato:

```
{DIR_SAIDA}/
├── png/   600 dpi
├── pdf/   vetorial, fontes embutidas
├── svg/   vetorial, texto editável
├── html/  gráficos interativos (Plotly)
└── <prefixo>_dados.csv   ← CSV ÚNICO com todos os dados
```

O CSV é **um só**, em formato longo: a coluna `tabela` diz de que tabela vem
cada linha; as colunas que não se aplicam ficam vazias. As linhas iniciadas por
`#` registram data, hora, versões e parâmetros. Para ler:
`pd.read_csv(arquivo, comment="#")`.

| `tabela` | conteúdo |
|---|---|
| `perfil` | S² médio ± SEM e IC por resíduo (e o experimental, se houver o CSV) |
| `por_replica` | S² por réplica (média dos blocos) e o da produção inteira |
| `por_bloco` | S² de cada bloco de `BLOCO_NS` |
| `estatistica` | S² médio da proteína por réplica e `entre_replicas` |
| `comparacao_experimental` | DM × RMN: média, desvio em incertezas e, com dados por resíduo, Pearson, Spearman e RMSD |

In [ ]:
FORMATOS = ("png", "pdf", "svg")
DPI = 600
FUNDO_TRANSPARENTE = False
PLOTLY_JS = "cdn"                  # "cdn" = HTML leve; True = embute o plotly.js (abre offline)
EXPORTAR_PLOTLY_ESTATICO = False   # True = também PNG/PDF/SVG do Plotly (precisa do kaleido)

FIGURAS = {"s2_perfil": fig_s2}
INTERATIVOS = {"s2_perfil": fig_interativo}
TABELAS = {"perfil": df_perfil, "por_replica": df_s2, "por_bloco": df_blocos, "estatistica": df_stats,
          "comparacao_experimental": df_comparacao}
PARAMETROS = {"descarte_ns": DESCARTE_NS, "dt_analise_ps": DT_ANALISE_PS, "bloco_ns": BLOCO_NS,
              "s2_medio_exp": S2_MEDIO_EXP, "arquivo_s2_exp": ARQUIVO_S2_EXP}

# ── Figuras ─────────────────────────────────────────────────────────────────
for nome, fig in FIGURAS.items():
    for ext in FORMATOS:
        (DIR_SAIDA / ext).mkdir(parents=True, exist_ok=True)
        fig.savefig(DIR_SAIDA / ext / f"{nome}.{ext}", dpi=DPI, bbox_inches="tight",
                    transparent=FUNDO_TRANSPARENTE)
for nome, fig in INTERATIVOS.items():
    (DIR_SAIDA / "html").mkdir(parents=True, exist_ok=True)
    fig.write_html(DIR_SAIDA / "html" / f"{nome}.html", include_plotlyjs=PLOTLY_JS,
                   config={"displaylogo": False, "toImageButtonOptions": {"format": "svg"}})
    if EXPORTAR_PLOTLY_ESTATICO:
        for ext in FORMATOS:
            try:
                fig.write_image(DIR_SAIDA / ext / f"{nome}_interativo.{ext}", scale=3)
            except Exception as erro:
                print(f"[aviso] {nome}.{ext} do Plotly não exportado: {erro}")
                break

# ── CSV único ───────────────────────────────────────────────────────────────
DIR_SAIDA.mkdir(parents=True, exist_ok=True)
partes = []
for nome, tabela in TABELAS.items():
    t = tabela.copy()
    t.insert(0, "tabela", nome)
    partes.append(t)
dados = pd.concat(partes, ignore_index=True, sort=False)
arquivo_csv = DIR_SAIDA / f"{PREFIXO}_dados.csv"
with open(arquivo_csv, "w", encoding="utf-8", newline="") as f:
    f.write(f"# {TITULO}\n")
    f.write(f"# gerado em {INICIO:%Y-%m-%d %H:%M:%S} | sistema {SISTEMA} | réplicas {REPLICAS}\n")
    f.write("# versoes: " + "; ".join(f"{k} {v}" for k, v in VERSOES.items()) + "\n")
    f.write("# parametros: " + "; ".join(f"{k} = {v}" for k, v in PARAMETROS.items()) + "\n")
    f.write("# tabelas: " + ", ".join(f"{k} ({len(v)} linhas)" for k, v in TABELAS.items()) + "\n")
    dados.to_csv(f, index=False)

# ── O que foi gravado ───────────────────────────────────────────────────────
df_exportados = pd.DataFrame([{"pasta": str(p.parent.relative_to(DIR_SAIDA)), "arquivo": p.name,
                               "tamanho_kB": round(p.stat().st_size / 1024, 1)}
                              for p in sorted(DIR_SAIDA.rglob("*")) if p.is_file()])
print(f"Resultados em: {DIR_SAIDA.resolve()}")
print(f"Duração da execução: {datetime.now() - INICIO}")
df_exportados